# JAMFUSE — Jensen–Agent Markov Fusion Engine
## Lightweight Bayesian Cross-Evidence Intelligence for Detecting Covert Data Exfiltration in Encrypted Traffic and Autonomous Agent Systems

**Notebook scope:** Stage 1–6, including data ingestion, encrypted-flow fingerprinting (ETAF), agent Markov/SPRT monitoring (AABM), dependence-aware Bayesian cross-evidence fusion (CEIF), interpretable decisions, and experimental evaluation.

### Research integrity and dataset boundaries
- CIC-IDS2017/Kaggle flow labels identify *network intrusion vs benign traffic*. They do **not** by themselves prove data exfiltration. ETAF is evaluated on intrusion/anomaly labels.
- AgentDojo is an interactive benchmark environment, not a ready-made timestamped network-flow dataset. Export/instrument agent tool, file, API and network events to the schema below. Its benchmark can be run independently; use the resulting trace logs only after mapping them into this schema.
- Controlled scenario rows created in this notebook are explicitly marked `source="CONTROLLED_SIMULATION"` and must never be reported as real-world observations. They are useful for software validation and controlled experiments only.
- Cross-evidence exfiltration evaluation is reported only when actual labelled aligned exfiltration events or explicitly labelled controlled scenarios exist.
- The notebook does not inspect encrypted payload contents. Flow features are metadata/statistics only.

**Colab:** Runtime → Change runtime type → CPU is sufficient for the default lightweight models. Mount Drive and set `FLOW_DATA_DIR` and `AGENT_LOG_PATH` below.

## Stage 0 — Environment and reproducibility

In [ ]:
import os, re, glob, json, time, math, warnings, hashlib, platform
from pathlib import Path
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.spatial.distance import jensenshannon
from scipy.special import expit, logit
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier, IsolationForest
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix, classification_report,
    precision_recall_curve, roc_curve)
from sklearn.calibration import CalibratedClassifierCV
import joblib

SEED = 42
np.random.seed(SEED)
RANDOM_STATE = SEED
MAX_FLOW_ROWS = 400_000       # lower this on RAM-limited Colab runtimes
TEST_SIZE = 0.20
TIME_TOLERANCE_SECONDS = 300  # maximum agent-to-flow timestamp difference
OUTPUT_DIR = Path("/content/JAMFUSE_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Python:", platform.python_version())
print("Pandas:", pd.__version__)
print("Output:", OUTPUT_DIR)

## Stage 1 — Open-source dataset collection and preparation

### 1A. Mount Google Drive and configure data locations
Place CIC-IDS2017 CSV files (or the Kaggle `training-flow.csv` / `test-flow.csv` files) in one folder. Provide an AgentDojo-derived event CSV separately if available.

Expected agent-event columns:
- `timestamp` — parseable datetime
- `action` — tool/file/API/network action name or normalized action
- `session_id` — optional but strongly recommended
- `label` — optional binary ground truth (0 benign, 1 suspicious/exfiltration)
- Optional: `event_type`, `tool_name`, `resource`, `direction`, `bytes_out`, `scenario_id`

For best temporal fusion, include `session_id` or a reliable flow key in both flow and agent logs. Timestamp-only nearest-neighbour alignment is weaker and is explicitly marked as such.

In [ ]:
# In Colab this mounts Drive. In a local Jupyter environment it safely skips.
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print("Google Drive mount skipped:", type(e).__name__)

# EDIT THESE PATHS
FLOW_DATA_DIR = "/content/drive/MyDrive/JAMFUSE/CIC-IDS2017"
AGENT_LOG_PATH = "/content/drive/MyDrive/JAMFUSE/agentdojo_events.csv"
# Optional: Kaggle API download. Set True only after uploading kaggle.json to Colab.
USE_KAGGLE_DOWNLOAD = False
KAGGLE_DATASET = "chethuhn/network-intrusion-dataset"
KAGGLE_DOWNLOAD_DIR = "/content/JAMFUSE_kaggle"

if USE_KAGGLE_DOWNLOAD:
    os.makedirs("/root/.kaggle", exist_ok=True)
    assert os.path.exists("/root/.kaggle/kaggle.json"), "Upload kaggle.json to /root/.kaggle first."
    os.system(f"kaggle datasets download -d {KAGGLE_DATASET} -p {KAGGLE_DOWNLOAD_DIR} --unzip")
    FLOW_DATA_DIR = KAGGLE_DOWNLOAD_DIR

print("Flow directory exists:", os.path.isdir(FLOW_DATA_DIR))
print("Agent log exists:", os.path.isfile(AGENT_LOG_PATH))

### 1B. Load and standardize CIC/Kaggle flow records
The loader handles common CICFlowMeter CSV naming variants and common Kaggle train/test flow CSV names. It removes whitespace/BOM from column names, coerces numeric features, removes infinite values, and creates `traffic_label` from the dataset's `Label` column. IP addresses, flow IDs and ports are not used as model features by default to reduce leakage and overfitting.

In [ ]:
def clean_colname(c):
    return re.sub(r"\\s+", " ", str(c).replace("\\ufeff", "")).strip()

def find_label_column(columns):
    for c in columns:
        if clean_colname(c).lower() in {"label", "class", "attack", "target"}:
            return c
    return None

def load_flow_csvs(folder, max_rows=MAX_FLOW_ROWS):
    files = []
    for pattern in ("**/*.csv", "**/*.CSV"):
        files.extend(glob.glob(os.path.join(folder, pattern), recursive=True))
    files = sorted(set(files))
    if not files:
        raise FileNotFoundError(
            f"No CSV flow files found in {folder}. Download CIC-IDS2017 CSVs or set FLOW_DATA_DIR."
        )
    frames, remaining = [], max_rows
    for fp in files:
        if remaining <= 0:
            break
        try:
            # Read in chunks so a large CIC CSV does not exhaust Colab RAM.
            parts = []
            for chunk in pd.read_csv(fp, low_memory=False, chunksize=100_000, encoding_errors="replace"):
                chunk.columns = [clean_colname(c) for c in chunk.columns]
                take = min(len(chunk), remaining)
                if take:
                    parts.append(chunk.iloc[:take].copy())
                    remaining -= take
                if remaining <= 0:
                    break
            if parts:
                d = pd.concat(parts, ignore_index=True)
                d["source_file"] = os.path.basename(fp)
                frames.append(d)
                print(os.path.basename(fp), "loaded rows:", len(d))
        except Exception as e:
            print("Skipped", fp, "->", str(e)[:180])
    if not frames:
        raise RuntimeError("CSV files were found but none could be read.")
    df = pd.concat(frames, ignore_index=True)
    df.columns = [clean_colname(c) for c in df.columns]
    label_col = find_label_column(df.columns)
    if label_col is None:
        raise ValueError("No Label/Class/Attack/Target column found in flow CSVs.")
    df["raw_label"] = df[label_col].astype(str).str.strip()
    df["traffic_label"] = (~df["raw_label"].str.lower().isin(
        ["benign", "normal", "0", "normal traffic"])).astype(int)
    # Preserve the original flow timestamp when available.
    time_col = next((c for c in df.columns if c.lower() in {"timestamp", "flow start", "flow start time", "starttime"}), None)
    df["flow_timestamp"] = pd.to_datetime(df[time_col], errors="coerce", dayfirst=False) if time_col else pd.NaT
    return df

flow_df = load_flow_csvs(FLOW_DATA_DIR)
print("Flow table:", flow_df.shape)
display(flow_df[["raw_label", "traffic_label"]].value_counts().head(20).rename("rows").to_frame())

### 1C. Optional AgentDojo event instrumentation / import
Export your own AgentDojo traces into the event schema. The cell below imports a CSV if present and normalizes action categories. It does not fabricate AgentDojo observations.

In [ ]:
ACTION_MAP = {
    "file_read": "FILE_READ", "read_file": "FILE_READ", "file_access": "FILE_READ",
    "file_write": "FILE_WRITE", "write_file": "FILE_WRITE",
    "tool": "TOOL_CALL", "tool_call": "TOOL_CALL", "invoke_tool": "TOOL_CALL",
    "api": "API_REQUEST", "api_request": "API_REQUEST", "http_request": "API_REQUEST",
    "network": "EXTERNAL_TRANSFER", "external_transfer": "EXTERNAL_TRANSFER",
    "send": "EXTERNAL_TRANSFER", "email_send": "EXTERNAL_TRANSFER",
    "search": "TOOL_CALL", "browser": "TOOL_CALL"
}

def normalize_action(x):
    s = str(x).strip().lower().replace("-", "_").replace(" ", "_")
    if s in ACTION_MAP:
        return ACTION_MAP[s]
    if any(k in s for k in ["read", "open", "download", "get_file"]): return "FILE_READ"
    if any(k in s for k in ["write", "delete", "modify", "upload_file"]): return "FILE_WRITE"
    if any(k in s for k in ["send", "post", "upload", "external", "egress"]): return "EXTERNAL_TRANSFER"
    if any(k in s for k in ["api", "http", "request"]): return "API_REQUEST"
    return "TOOL_CALL"

agent_df = pd.DataFrame()
if os.path.isfile(AGENT_LOG_PATH):
    agent_df = pd.read_csv(AGENT_LOG_PATH)
    agent_df.columns = [clean_colname(c).lower().replace(" ", "_") for c in agent_df.columns]
    required = {"timestamp", "action"}
    if not required.issubset(agent_df.columns):
        raise ValueError(f"Agent log must contain {required}; got {agent_df.columns.tolist()}")
    agent_df["timestamp"] = pd.to_datetime(agent_df["timestamp"], errors="coerce", utc=True)
    agent_df["action_state"] = agent_df["action"].map(normalize_action)
    if "session_id" not in agent_df: agent_df["session_id"] = np.nan
    if "label" not in agent_df: agent_df["label"] = np.nan
    agent_df["label"] = pd.to_numeric(agent_df["label"], errors="coerce")
    agent_df["source"] = "AGENT_LOG"
    agent_df = agent_df.dropna(subset=["timestamp", "action_state"]).reset_index(drop=True)
else:
    print("No AgentDojo event CSV supplied. AABM training requires real event logs; controlled simulation is optional in Stage 1E.")
print("Agent events:", agent_df.shape)
if len(agent_df): display(agent_df.head())

### 1D. Controlled scenario generation — optional software-validation data
This cell can create transparent, labelled toy traces to test the complete pipeline when no agent event export exists. These rows are **not real AgentDojo measurements** and are excluded from real-data claims. Disable the cell for experiments that must use only observed data.

In [ ]:
ENABLE_CONTROLLED_SIMULATION = False
SIMULATION_SESSIONS = 1200

def make_controlled_agent_scenarios(n_sessions=SIMULATION_SESSIONS, seed=SEED):
    rng = np.random.default_rng(seed)
    rows = []
    benign = ["TOOL_CALL", "FILE_READ", "TOOL_CALL", "API_REQUEST", "TOOL_CALL"]
    suspicious = ["TOOL_CALL", "FILE_READ", "FILE_READ", "FILE_WRITE", "EXTERNAL_TRANSFER"]
    base = pd.Timestamp("2025-01-01T00:00:00Z")
    for i in range(n_sessions):
        y = int(i % 2 == 1)
        seq = suspicious if y else benign
        start = base + pd.Timedelta(seconds=int(i * 600))
        for j, act in enumerate(seq):
            # Controlled timestamps are generated solely for pipeline validation.
            rows.append({"timestamp": start + pd.Timedelta(seconds=j*7 + int(rng.integers(0,3))),
                         "action": act, "action_state": act, "session_id": f"SIM_{i:06d}",
                         "label": y, "source": "CONTROLLED_SIMULATION",
                         "scenario_id": f"SIM_{i:06d}"})
    return pd.DataFrame(rows)

if ENABLE_CONTROLLED_SIMULATION:
    sim_agent_df = make_controlled_agent_scenarios()
    agent_df = pd.concat([agent_df, sim_agent_df], ignore_index=True) if len(agent_df) else sim_agent_df
    print("Controlled simulation rows:", len(sim_agent_df))
else:
    print("Controlled simulation disabled.")

### 1E. Data quality, provenance and labels

In [ ]:
def data_quality_report(df, name):
    print(f"\\n{name}: {df.shape}")
    print("Duplicate rows:", int(df.duplicated().sum()))
    print("Missing fraction (top 10):")
    display(df.isna().mean().sort_values(ascending=False).head(10).to_frame("missing_fraction"))
    return {"dataset": name, "rows": len(df), "columns": df.shape[1],
            "duplicate_rows": int(df.duplicated().sum())}

quality = [data_quality_report(flow_df, "flow_df")]
if len(agent_df): quality.append(data_quality_report(agent_df, "agent_df"))
pd.DataFrame(quality)

## Stage 2 — ETAF: Encrypted Traffic Anomaly Fingerprinting (Contribution 1)

ETAF uses flow metadata only. The feature vector emphasizes packet-size statistics, flow duration, inter-arrival time, directional volume, and connection-rate features. Jensen–Shannon divergence is computed between each flow's normalized metadata fingerprint and the benign training reference distribution. A robust classifier combines these signals; adaptive thresholding is fitted on training data only.

In [ ]:
EXCLUDE_TOKENS = ["label", "target", "attack", "class", "source_file", "raw_label",
                  "timestamp", "flow id", "src ip", "dst ip", "source ip", "destination ip",
                  "src port", "dst port", "source port", "destination port", "protocol"]
PREFERRED_HINTS = ["flow duration", "packet length", "packet size", "iat", "fwd", "bwd",
                   "flow bytes", "flow packets", "total length", "total packets",
                   "active", "idle", "down/up", "header length", "flag count"]

def choose_numeric_flow_features(df):
    cols = []
    for c in df.columns:
        cl = c.lower()
        if any(t in cl for t in EXCLUDE_TOKENS): continue
        if pd.api.types.is_numeric_dtype(df[c]):
            cols.append(c)
    if not cols:
        raise ValueError("No numeric flow metadata columns found.")
    # Keep all available numeric metadata columns except identifiers and labels.
    return cols

flow_features = choose_numeric_flow_features(flow_df)
X_flow = flow_df[flow_features].replace([np.inf, -np.inf], np.nan)
y_flow = flow_df["traffic_label"].astype(int).values
print("ETAF feature count:", len(flow_features))
print("Traffic class distribution:", Counter(y_flow))

# Stratified split. If source_file exists, it is retained for audit but never used as a feature.
idx_train, idx_test = train_test_split(np.arange(len(flow_df)), test_size=TEST_SIZE,
    random_state=SEED, stratify=y_flow)
Xf_train, Xf_test = X_flow.iloc[idx_train], X_flow.iloc[idx_test]
yf_train, yf_test = y_flow[idx_train], y_flow[idx_test]

etaf_preprocess = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", RobustScaler(quantile_range=(10, 90)))
])
Xf_train_s = etaf_preprocess.fit_transform(Xf_train)
Xf_test_s = etaf_preprocess.transform(Xf_test)

etaf_model = HistGradientBoostingClassifier(max_iter=140, learning_rate=0.08,
    max_leaf_nodes=15, l2_regularization=1.0, random_state=SEED)
t0 = time.perf_counter()
etaf_model.fit(Xf_train_s, yf_train)
etaf_train_seconds = time.perf_counter() - t0
etaf_prob_test = etaf_model.predict_proba(Xf_test_s)[:, 1]
etaf_pred_test = (etaf_prob_test >= 0.5).astype(int)
print("ETAF training seconds:", round(etaf_train_seconds, 3))

In [ ]:
# Robust statistical fingerprint reference: benign training median/IQR.
Xf_train_raw = Xf_train.copy()
benign_train = Xf_train_raw.loc[yf_train == 0]
ref_median = benign_train.median(numeric_only=True)
ref_iqr = (benign_train.quantile(.75) - benign_train.quantile(.25)).replace(0, 1).fillna(1)

def row_fingerprint_js(row, reference=ref_median, scale=ref_iqr):
    vals = pd.to_numeric(row.reindex(reference.index), errors="coerce")
    z = ((vals - reference) / scale).replace([np.inf, -np.inf], np.nan).fillna(0).values
    # Map robust deviations into a nonnegative histogram, then compare to centered reference.
    observed = np.abs(z)
    p = observed + 1e-8
    p = p / p.sum()
    q = np.ones_like(p) / len(p)
    return float(jensenshannon(p, q, base=2) ** 2)

js_scores_test = np.array([row_fingerprint_js(r) for _, r in Xf_test.iterrows()])
# Adaptive anomaly cutoff is chosen using benign training scores (95th percentile).
js_scores_benign_train = np.array([row_fingerprint_js(r) for _, r in benign_train.iterrows()])
js_threshold = float(np.quantile(js_scores_benign_train, 0.95))
js_anomaly_test = (js_scores_test > js_threshold).astype(int)

# ETAF combines supervised probability and distributional deviation.
js_scaled_test = np.clip(js_scores_test / max(js_threshold, 1e-6), 0, 3) / 3
etaf_score_test = np.clip(0.75 * etaf_prob_test + 0.25 * js_scaled_test, 0, 1)
# Fit the adaptive cutoff using benign TRAINING rows only (no test-label leakage).
train_prob = etaf_model.predict_proba(Xf_train_s)[:, 1]
js_scores_benign_train_all = np.array([row_fingerprint_js(r) for _, r in benign_train.iterrows()])
train_benign_prob = train_prob[yf_train == 0]
train_benign_js_scaled = np.clip(js_scores_benign_train_all / max(js_threshold, 1e-6), 0, 3) / 3
train_benign_etaf_score = np.clip(0.75 * train_benign_prob + 0.25 * train_benign_js_scaled, 0, 1)
etaf_adaptive_threshold = float(np.quantile(train_benign_etaf_score, 0.95))
etaf_pred_test = (etaf_score_test >= etaf_adaptive_threshold).astype(int)

print("JS reference threshold:", round(js_threshold, 5))
print("ETAF adaptive decision threshold:", etaf_adaptive_threshold)

## Stage 3 — AABM: Autonomous Agent Behaviour Monitoring (Contribution 2)

Agent actions are mapped into finite states. A first-order Markov model learns normal transition probabilities from benign training sequences. Transition surprisal is accumulated with a Sequential Probability Ratio Test (SPRT). When labelled benign/suspicious traces are available, a lightweight classifier learns from per-session Markov and event-count features. If no agent log exists, AABM is correctly marked unavailable rather than inventing real events.

In [ ]:
STATES = ["TOOL_CALL", "FILE_READ", "FILE_WRITE", "API_REQUEST", "EXTERNAL_TRANSFER", "OTHER"]
STATE_TO_ID = {s:i for i,s in enumerate(STATES)}

def state_of(x):
    s = normalize_action(x)
    return s if s in STATE_TO_ID else "OTHER"

def build_transition_model(events, benign_only=True, alpha=0.5):
    counts = np.full((len(STATES), len(STATES)), alpha, dtype=float)
    d = events.copy()
    if benign_only and "label" in d:
        d = d[pd.to_numeric(d["label"], errors="coerce").fillna(0).astype(int) == 0]
    group_col = "session_id" if d["session_id"].notna().any() else None
    groups = d.groupby(group_col, dropna=False) if group_col else [(0, d.sort_values("timestamp"))]
    for _, g in groups:
        seq = g.sort_values("timestamp")["action_state"].map(state_of).tolist()
        for a, b in zip(seq[:-1], seq[1:]):
            counts[STATE_TO_ID[a], STATE_TO_ID[b]] += 1
    return counts / counts.sum(axis=1, keepdims=True)

def session_markov_features(g, transition_matrix):
    seq = g.sort_values("timestamp")["action_state"].map(state_of).tolist()
    ids = [STATE_TO_ID[x] for x in seq]
    trans = [transition_matrix[a,b] for a,b in zip(ids[:-1], ids[1:])]
    surprisal = [-math.log(max(p, 1e-9)) for p in trans]
    counts = Counter(seq)
    out = {f"count_{s}": counts.get(s, 0) for s in STATES}
    out.update({"n_events": len(seq), "mean_transition_surprisal": np.mean(surprisal) if surprisal else 0,
                "max_transition_surprisal": np.max(surprisal) if surprisal else 0,
                "external_after_file": sum(1 for a,b in zip(seq[:-1],seq[1:]) if a in ["FILE_READ","FILE_WRITE"] and b=="EXTERNAL_TRANSFER"),
                "transition_count": len(trans)})
    return out

def sprt_score(g, p0, p1, threshold_a=2.944, threshold_b=-2.944):
    seq = g.sort_values("timestamp")["action_state"].map(state_of).tolist()
    llr, crossed = 0.0, False
    increments = []
    for a,b in zip(seq[:-1], seq[1:]):
        i,j = STATE_TO_ID[a], STATE_TO_ID[b]
        inc = math.log(max(p1[i,j],1e-9) / max(p0[i,j],1e-9))
        llr += inc
        increments.append(llr)
        if llr >= threshold_a: crossed = True
        if llr <= threshold_b: break
    return float(llr), int(crossed), len(increments)

aabm_available = len(agent_df) > 0
aabm_session_df = pd.DataFrame()
if aabm_available:
    agent_df["action_state"] = agent_df["action_state"].fillna(agent_df["action"].map(state_of)).map(state_of)
    agent_df["session_id"] = agent_df["session_id"].fillna("UNKEYED_SESSION")
    agent_df["label_num"] = pd.to_numeric(agent_df["label"], errors="coerce")
    p0 = build_transition_model(agent_df, benign_only=True)
    # Alternative distribution from explicitly labelled suspicious sequences, if present.
    if agent_df["label_num"].eq(1).any():
        # Estimate the alternative transition model from labelled suspicious sessions only.
        suspicious_events = agent_df[agent_df["label_num"] == 1].copy()
        p1 = build_transition_model(suspicious_events, benign_only=False, alpha=0.5)
    else:
        p1 = np.full_like(p0, 1/len(STATES))
    session_rows = []
    for sid, g in agent_df.groupby("session_id"):
        feats = session_markov_features(g, p0)
        llr, alarm, ntrans = sprt_score(g, p0, p1)
        known = g["label_num"].dropna()
        feats.update({"session_id":sid, "sprt_llr":llr, "sprt_alarm":alarm,
                      "agent_label": int(known.max()) if len(known) else np.nan,
                      "event_source": ",".join(sorted(g["source"].dropna().astype(str).unique()))})
        session_rows.append(feats)
    aabm_session_df = pd.DataFrame(session_rows)
    print("AABM sessions:", aabm_session_df.shape)
    display(aabm_session_df.head())
else:
    print("AABM unavailable: provide timestamped agent events to train/evaluate this stage.")

In [ ]:
aabm_feature_cols = [c for c in aabm_session_df.columns if c.startswith("count_") or c in
    ["n_events","mean_transition_surprisal","max_transition_surprisal",
     "external_after_file","transition_count","sprt_llr","sprt_alarm"]]
aabm_model, aabm_metrics_available = None, False
if aabm_available and aabm_session_df["agent_label"].notna().sum() >= 20 and aabm_session_df["agent_label"].nunique() == 2:
    valid = aabm_session_df.dropna(subset=["agent_label"]).copy()
    Xa = valid[aabm_feature_cols].replace([np.inf,-np.inf],np.nan)
    ya = valid["agent_label"].astype(int)
    Xa_tr, Xa_te, ya_tr, ya_te = train_test_split(Xa, ya, test_size=.25, random_state=SEED, stratify=ya)
    aabm_model = Pipeline([("imputer",SimpleImputer(strategy="median")),
        ("scaler",RobustScaler()),("clf",LogisticRegression(max_iter=1000,class_weight="balanced",random_state=SEED))])
    aabm_model.fit(Xa_tr, ya_tr)
    aabm_prob_eval = aabm_model.predict_proba(Xa_te)[:,1]
    aabm_pred_eval = (aabm_prob_eval >= .5).astype(int)
    aabm_metrics_available = True
    print(classification_report(ya_te, aabm_pred_eval, zero_division=0))
else:
    print("AABM supervised evaluation skipped: requires >=20 labelled sessions and both classes.")

## Stage 4 — CEIF: Cross-Evidence Exfiltration Identification and Fusion (Contribution 3)

### Dependence-aware Bayesian fusion
A naive multiplication of traffic and agent likelihoods assumes conditional independence and can become overconfident. JAMFUSE uses calibrated evidence probabilities, converts them to log-odds, and applies a dependence discount `rho` to the second evidence channel. `rho` is estimated from the training evidence correlation and bounded to a conservative range. This is an approximation, not a full causal dependence model; report it as such.

In [ ]:
def safe_auc(y, p):
    return roc_auc_score(y,p) if len(np.unique(y)) == 2 else np.nan

def metric_dict(y, p, threshold=.5):
    y = np.asarray(y).astype(int); p = np.asarray(p).astype(float)
    pred = (p >= threshold).astype(int)
    return {
        "Accuracy": accuracy_score(y,pred),
        "Precision": precision_score(y,pred,zero_division=0),
        "Recall": recall_score(y,pred,zero_division=0),
        "F1": f1_score(y,pred,zero_division=0),
        "AUROC": safe_auc(y,p),
        "AUPRC": average_precision_score(y,p) if len(np.unique(y))==2 else np.nan,
        "FPR": (confusion_matrix(y,pred,labels=[0,1])[0,1] /
                max(1,confusion_matrix(y,pred,labels=[0,1])[0].sum()))
    }

def calibrate_probability(p, eps=1e-5):
    return np.clip(np.asarray(p,dtype=float), eps, 1-eps)

def bayesian_dependence_fusion(p_traffic, p_agent, rho=0.5, prior=0.05):
    pt, pa = calibrate_probability(p_traffic), calibrate_probability(p_agent)
    prior_logodds = logit(np.clip(prior,1e-5,1-1e-5))
    # Evidence log-Bayes factors relative to the prior.
    lt = logit(pt) - prior_logodds
    la = logit(pa) - prior_logodds
    fused_logodds = prior_logodds + lt + rho*la
    return expit(fused_logodds)

# Define a generic event-level alignment: nearest flow timestamp within tolerance.
# If both sources carry session_id, exact session matching takes precedence.
def align_agent_to_flows(flows, agents, tolerance_seconds=TIME_TOLERANCE_SECONDS):
    if len(agents)==0:
        return pd.DataFrame()
    f = flows.copy()
    f["flow_row_id"] = np.arange(len(f))
    f["flow_timestamp"] = pd.to_datetime(f["flow_timestamp"], errors="coerce", utc=True)
    a = agents.copy()
    a["timestamp"] = pd.to_datetime(a["timestamp"], errors="coerce", utc=True)
    a = a.dropna(subset=["timestamp"])
    # Exact session alignment when the flow table actually contains a session identifier.
    flow_sid = next((c for c in f.columns if c.lower() in {"session_id","session","scenario_id"}),None)
    if flow_sid and "session_id" in a:
        merged = f.merge(a, left_on=flow_sid, right_on="session_id", how="inner", suffixes=("_flow","_agent"))
        if len(merged): return merged
    f = f.dropna(subset=["flow_timestamp"]).sort_values("flow_timestamp")
    a = a.sort_values("timestamp")
    if len(f)==0:
        print("No usable flow timestamps; cannot temporal-align agent events.")
        return pd.DataFrame()
    merged = pd.merge_asof(a, f, left_on="timestamp", right_on="flow_timestamp",
        direction="nearest", tolerance=pd.Timedelta(seconds=tolerance_seconds),
        suffixes=("_agent","_flow"))
    return merged.dropna(subset=["flow_row_id"])

aligned_df = align_agent_to_flows(flow_df, agent_df) if aabm_available else pd.DataFrame()
print("Aligned agent-flow rows:", len(aligned_df))
if len(aligned_df):
    print("Alignment mode:", "session-key or nearest timestamp")
    print("Sources:", aligned_df.get("source",pd.Series(dtype=str)).value_counts().to_dict())

### 4A. Build aligned cross-evidence evaluation table
A true exfiltration label must come from an explicit exfiltration annotation/scenario, not merely from a generic CIC intrusion label. The default `exfiltration_label` is populated only from an agent `label` field explicitly supplied by the user. When a separate aligned ground-truth column exists, configure it below.

In [ ]:
# Configure the agent label semantics. Use 1 only for confirmed/simulated exfiltration or suspicious
# agent activity as defined by your experiment; document the exact operational definition.
EXFIL_LABEL_COLUMN = "label"  # edit if your exported log uses another name
ceif_df = pd.DataFrame()
if len(aligned_df):
    # Agent suspicion from Markov feature score or labelled event; do not silently infer exfiltration
    if "label" in aligned_df.columns:
        aligned_df["agent_label_numeric"] = pd.to_numeric(aligned_df["label"], errors="coerce")
    else:
        aligned_df["agent_label_numeric"] = np.nan
    # Map ETAF score by original flow row index when available.
    aligned_df["traffic_evidence"] = aligned_df["traffic_label"].astype(float)
    aligned_df["agent_evidence"] = aligned_df["agent_label_numeric"]
    # Keep only rows with explicit labels for a valid CEIF supervised evaluation.
    ceif_df = aligned_df.dropna(subset=["agent_evidence"]).copy()
    ceif_df["exfiltration_label"] = ceif_df["agent_evidence"].astype(int).clip(0,1)
    print("Explicitly labelled aligned rows:", len(ceif_df))
else:
    print("CEIF supervised data unavailable: no aligned agent/flow evidence.")

In [ ]:
ceif_model = None
ceif_metrics_available = False
ceif_results = pd.DataFrame()
if len(ceif_df) >= 30 and ceif_df["exfiltration_label"].nunique()==2:
    # Create per-flow evidence scores using trained ETAF probabilities.
    flow_prob_lookup = pd.Series(etaf_prob_test, index=idx_test)
    ceif_df["flow_row_id"] = pd.to_numeric(ceif_df["flow_row_id"],errors="coerce")
    ceif_df["traffic_prob"] = ceif_df["flow_row_id"].map(flow_prob_lookup).fillna(
        ceif_df["traffic_evidence"] * .8 + .1)
    # Agent probability: use calibrated supervised model when available, otherwise Markov/SRPT
    # evidence is transformed monotonically and clearly identified as a heuristic score.
    if aabm_model is not None:
        # session-level model probabilities are used as a mapping by session id
        valid_sess = aabm_session_df.dropna(subset=["agent_label"]).copy()
        if len(valid_sess):
            valid_sess["agent_prob"] = aabm_model.predict_proba(valid_sess[aabm_feature_cols])[:,1]
            apmap = valid_sess.set_index("session_id")["agent_prob"]
            ceif_df["agent_prob"] = ceif_df["session_id"].map(apmap).fillna(.5)
        else: ceif_df["agent_prob"] = .5
    else:
        ceif_df["agent_prob"] = expit(pd.to_numeric(ceif_df["agent_evidence"],errors="coerce").fillna(0)*2-1)
    rho = 0.5
    if len(ceif_df)>10:
        corr = pd.Series(ceif_df["traffic_prob"]).corr(pd.Series(ceif_df["agent_prob"]))
        rho = float(np.clip(1-abs(corr if pd.notna(corr) else 0), .15, .85))
    ceif_df["ceif_probability"] = bayesian_dependence_fusion(
        ceif_df["traffic_prob"].values, ceif_df["agent_prob"].values, rho=rho, prior=.05)
    y_ceif = ceif_df["exfiltration_label"].values
    # Cost-sensitive threshold: choose threshold maximizing F1 with a mild false-positive penalty.
    precision, recall, thresholds = precision_recall_curve(y_ceif, ceif_df["ceif_probability"])
    if len(thresholds):
        f1s = 2*precision[:-1]*recall[:-1]/np.maximum(precision[:-1]+recall[:-1],1e-9)
        best_idx = int(np.argmax(f1s - .10*(1-precision[:-1])))
        ceif_threshold = float(thresholds[best_idx])
    else: ceif_threshold = .5
    ceif_results = ceif_df[["exfiltration_label","traffic_prob","agent_prob","ceif_probability"]].copy()
    ceif_metrics_available = True
    print("Dependence discount rho:", round(rho,3), "| decision threshold:", round(ceif_threshold,4))
    print(metric_dict(y_ceif, ceif_df["ceif_probability"], ceif_threshold))
else:
    rho, ceif_threshold = 0.5, 0.5
    print("CEIF evaluation skipped. Need >=30 aligned labelled rows and both classes.")

## Stage 5 — Integrated security decision and reporting

The four output states are:
1. `NORMAL`
2. `SUSPICIOUS_ENCRYPTED_COMMUNICATION`
3. `SUSPICIOUS_AGENT_BEHAVIOUR`
4. `LIKELY_COVERT_EXFILTRATION` (requires cross-evidence threshold)

The final report lists evidence features and scores. A high traffic anomaly score alone is not labelled as exfiltration.

In [ ]:
def integrated_security_decision(traffic_score, agent_score=None, fused_score=None,
                                 traffic_threshold=.5, agent_threshold=.5, fused_threshold=.5):
    ta = float(traffic_score) >= traffic_threshold
    aa = agent_score is not None and pd.notna(agent_score) and float(agent_score) >= agent_threshold
    if fused_score is not None and pd.notna(fused_score) and float(fused_score) >= fused_threshold:
        return "LIKELY_COVERT_EXFILTRATION", "HIGH"
    if ta and aa: return "SUSPICIOUS_CROSS_EVIDENCE", "HIGH"
    if ta: return "SUSPICIOUS_ENCRYPTED_COMMUNICATION", "MEDIUM"
    if aa: return "SUSPICIOUS_AGENT_BEHAVIOUR", "MEDIUM"
    return "NORMAL", "LOW"

# Produce a per-flow ETAF report for the held-out set.
security_report = flow_df.iloc[idx_test][["raw_label","traffic_label"]].copy()
security_report["etaf_probability"] = etaf_score_test
security_report["etaf_js_divergence"] = js_scores_test
security_report["security_class"] = [
    integrated_security_decision(s, traffic_threshold=etaf_adaptive_threshold)[0]
    for s in etaf_score_test
]
security_report["risk_level"] = [
    integrated_security_decision(s, traffic_threshold=etaf_adaptive_threshold)[1]
    for s in etaf_score_test
]
security_report["evidence"] = [
    "ETAF: robust flow metadata + JS divergence" if s >= etaf_adaptive_threshold else "No threshold-crossing ETAF evidence"
    for s in etaf_score_test
]
display(security_report.head(10))
security_report.to_csv(OUTPUT_DIR/"etaf_security_report.csv",index=False)

## Stage 6 — Experimental evaluation and validation

### 6A. ETAF metrics
Metrics include Accuracy, Precision, Recall, F1, AUROC, AUPRC, FPR, detection latency and throughput. Detection latency is measured as average per-row inference time on the held-out in-memory sample; it is not a network end-to-end latency.

In [ ]:
def evaluate_binary(y, p, threshold=.5, name="model"):
    m = metric_dict(y,p,threshold)
    m["Model"] = name
    return m

etaf_metrics = evaluate_binary(yf_test, etaf_score_test, etaf_adaptive_threshold, "ETAF")
t0=time.perf_counter()
_ = etaf_model.predict_proba(Xf_test_s)
inference_seconds=time.perf_counter()-t0
etaf_metrics["Detection_Latency_ms_per_flow"] = 1000*inference_seconds/max(1,len(Xf_test_s))
etaf_metrics["Throughput_flows_per_second"] = len(Xf_test_s)/max(inference_seconds,1e-9)
etaf_metrics["Training_Time_seconds"] = etaf_train_seconds
print(pd.DataFrame([etaf_metrics]).T)

### 6B. AABM metrics
Abnormal Sequence Detection Rate is reported as recall on the suspicious class. Sequence processing time is measured per session. If labels are missing or only one class exists, metrics are intentionally omitted.

In [ ]:
aabm_metrics = {}
if aabm_metrics_available:
    aabm_metrics = metric_dict(ya_te, aabm_prob_eval, .5)
    aabm_metrics["Abnormal_Sequence_Detection_Rate"] = aabm_metrics["Recall"]
    t0=time.perf_counter()
    _=aabm_model.predict_proba(Xa_te)
    sec=time.perf_counter()-t0
    aabm_metrics["Sequence_Processing_Time_ms"] = 1000*sec/max(1,len(Xa_te))
    display(pd.DataFrame([aabm_metrics],index=["AABM"]).T)
else:
    print("AABM metrics unavailable: labelled agent sessions from both classes are required.")

### 6C. CEIF metrics and cross-evidence improvement
CEIF reports exfiltration detection rate/recall, precision, Macro-F1 (binary macro average), AUROC, AUPRC and FPR. Compare traffic-only, agent-only and fused evidence on the **same aligned labelled rows**.

In [ ]:
comparison_rows = []
if ceif_metrics_available:
    y = ceif_df["exfiltration_label"].values
    for name, p in [
        ("Traffic-only", ceif_df["traffic_prob"].values),
        ("Agent-only", ceif_df["agent_prob"].values),
        ("JAMFUSE-CEIF", ceif_df["ceif_probability"].values)
    ]:
        mm = metric_dict(y,p,ceif_threshold if name=="JAMFUSE-CEIF" else .5)
        mm["Model"] = name
        mm["Macro-F1"] = f1_score(y,(p >= (ceif_threshold if name=="JAMFUSE-CEIF" else .5)).astype(int),average="macro")
        comparison_rows.append(mm)
    comparison_df = pd.DataFrame(comparison_rows).set_index("Model")
    display(comparison_df)
    comparison_df.to_csv(OUTPUT_DIR/"ceif_comparison_metrics.csv")
else:
    print("No valid CEIF metrics; supply real aligned labels or enable clearly marked controlled scenarios.")

### 6D. Ablation study
The ablation removes JS divergence, Markov/SPRT agent evidence, and dependence correction. It runs only when valid labelled aligned CEIF rows exist.

In [ ]:
if ceif_metrics_available:
    p_t = ceif_df["traffic_prob"].values
    p_a = ceif_df["agent_prob"].values
    p_full = ceif_df["ceif_probability"].values
    ablations = {
        "Full JAMFUSE": p_full,
        "Without agent evidence (ETAF only)": p_t,
        "Without traffic evidence (AABM only)": p_a,
        "Naive Bayesian fusion (rho=1)": bayesian_dependence_fusion(p_t,p_a,rho=1.0,prior=.05),
        "Dependence-discounted fusion": bayesian_dependence_fusion(p_t,p_a,rho=rho,prior=.05)
    }
    ablation_rows=[]
    for name,p in ablations.items():
        m=metric_dict(y,p,ceif_threshold if name=="Full JAMFUSE" else .5); m["Ablation"]=name
        ablation_rows.append(m)
    ablation_df=pd.DataFrame(ablation_rows).set_index("Ablation")
    display(ablation_df)
    ablation_df.to_csv(OUTPUT_DIR/"ablation_metrics.csv")
else:
    print("Ablation skipped: CEIF labels unavailable.")

### 6E. Statistical significance (paired bootstrap)
Bootstrap confidence intervals compare fused CEIF F1 against traffic-only F1 on the same labelled observations. This is a paired resampling estimate; it is not a substitute for repeated independent datasets.

In [ ]:
def paired_bootstrap_f1(y, p_a, p_b, threshold_a=.5, threshold_b=.5, n_boot=1000, seed=SEED):
    rng=np.random.default_rng(seed); y=np.asarray(y); p_a=np.asarray(p_a); p_b=np.asarray(p_b)
    diffs=[]
    for _ in range(n_boot):
        ix=rng.integers(0,len(y),len(y))
        if len(np.unique(y[ix]))<2: continue
        fa=f1_score(y[ix],p_a[ix]>=threshold_a,zero_division=0)
        fb=f1_score(y[ix],p_b[ix]>=threshold_b,zero_division=0)
        diffs.append(fa-fb)
    return np.quantile(diffs,[.025,.5,.975]) if diffs else [np.nan]*3

if ceif_metrics_available:
    ci=paired_bootstrap_f1(y_ceif, ceif_df["ceif_probability"].values,
        ceif_df["traffic_prob"].values, ceif_threshold, .5)
    print("F1 improvement (CEIF - traffic-only), 95% CI:", ci)
else:
    print("Bootstrap significance test skipped.")

### 6F. Cross-dataset validation
For a defensible cross-dataset experiment, place a second independently sourced flow dataset in a separate folder and apply the already-fitted ETAF preprocessing/model without refitting. Its label column must use benign/normal vs attack semantics. This measures transfer on network anomaly detection, not exfiltration.

In [ ]:
SECOND_FLOW_DATA_DIR = ""  # e.g. "/content/drive/MyDrive/JAMFUSE/CSE-CIC-IDS2018"
if SECOND_FLOW_DATA_DIR and os.path.isdir(SECOND_FLOW_DATA_DIR):
    cross_df=load_flow_csvs(SECOND_FLOW_DATA_DIR,max_rows=100_000)
    common=[c for c in flow_features if c in cross_df.columns]
    if len(common)==len(flow_features):
        Xcross=cross_df[flow_features].replace([np.inf,-np.inf],np.nan)
        Xcross_s=etaf_preprocess.transform(Xcross)
        pcross=etaf_model.predict_proba(Xcross_s)[:,1]
        ycross=cross_df["traffic_label"].values
        print("Cross-dataset ETAF:", metric_dict(ycross,pcross,.5))
    else:
        print("Feature mismatch. Shared features:",len(common),"of",len(flow_features),
              "- harmonize feature schemas before cross-dataset evaluation.")
else:
    print("Cross-dataset validation not run. Configure SECOND_FLOW_DATA_DIR.")

### 6G. Lightweight computational profile
Measures model artifact size, inference time, throughput and process memory where `psutil` is available. CPU utilization is sampled as a coarse process-level measurement.

In [ ]:
import pickle, sys
model_bundle = {"etaf_preprocess":etaf_preprocess,"etaf_model":etaf_model,
                "aabm_model":aabm_model,"flow_features":flow_features,
                "transition_matrix":p0 if aabm_available else None}
bundle_bytes=len(pickle.dumps(model_bundle))
try:
    import psutil
    proc=psutil.Process(os.getpid())
    memory_mb=proc.memory_info().rss/(1024**2)
    cpu_pct=psutil.cpu_percent(interval=0.5)
except Exception:
    memory_mb=np.nan; cpu_pct=np.nan

profile = pd.DataFrame([{
    "Model_Size_KB":bundle_bytes/1024,
    "Process_Memory_MB":memory_mb,
    "CPU_Utilization_percent_sample":cpu_pct,
    "ETAF_Inference_ms_per_flow":etaf_metrics["Detection_Latency_ms_per_flow"],
    "ETAF_Throughput_flows_per_second":etaf_metrics["Throughput_flows_per_second"],
    "ETAF_Training_Time_seconds":etaf_train_seconds
}])
display(profile)
profile.to_csv(OUTPUT_DIR/"lightweight_profile.csv",index=False)

### 6H. Plots and confusion matrices

In [ ]:
fig, axes = plt.subplots(1,2,figsize=(12,4))
cm=confusion_matrix(yf_test,etaf_pred_test,labels=[0,1])
sns.heatmap(cm,annot=True,fmt="d",cmap="Blues",ax=axes[0])
axes[0].set_title("ETAF Confusion Matrix"); axes[0].set_xlabel("Predicted"); axes[0].set_ylabel("Actual")
axes[1].hist(js_scores_test[yf_test==0],bins=40,alpha=.65,label="Benign")
axes[1].hist(js_scores_test[yf_test==1],bins=40,alpha=.65,label="Attack")
axes[1].axvline(js_threshold,linestyle="--",label="Benign 95th percentile")
axes[1].set_title("Jensen–Shannon Fingerprint Scores"); axes[1].legend()
plt.tight_layout(); plt.show()

if ceif_metrics_available:
    fpr,tpr,_=roc_curve(y_ceif,ceif_df["ceif_probability"])
    plt.figure(figsize=(5,4)); plt.plot(fpr,tpr,label=f"JAMFUSE AUROC={roc_auc_score(y_ceif,ceif_df['ceif_probability']):.3f}")
    plt.plot([0,1],[0,1],"--"); plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
    plt.title("CEIF ROC Curve"); plt.legend(); plt.tight_layout(); plt.show()

## Final export and reproducibility manifest

Exports include the ETAF per-flow report, available metric tables, feature schema, fitted lightweight models and a manifest recording the provenance caveats. Do not publish simulation-derived scores as real-world AgentDojo results.

In [ ]:
# Save trained artifacts locally for reproducibility (not required for inference in this notebook).
joblib.dump(model_bundle, OUTPUT_DIR/"jamfuse_lightweight_models.joblib")
with open(OUTPUT_DIR/"feature_dictionary.json","w") as f:
    json.dump({
        "flow_features":flow_features,
        "agent_states":STATES,
        "labels":{"traffic_label":"0=benign/normal, 1=attack/anomaly (CIC label mapping)",
                  "agent_label":"user-supplied event/session label; define its semantics",
                  "exfiltration_label":"explicitly annotated aligned exfiltration/suspicious event only"},
        "provenance":{"flow_dataset":"user-supplied CIC-IDS2017/Kaggle CSVs",
                      "agent_dataset":"user-supplied AgentDojo instrumentation CSV" if os.path.isfile(AGENT_LOG_PATH) else "not supplied",
                      "controlled_simulation_enabled":ENABLE_CONTROLLED_SIMULATION},
        "limitations":[
            "CIC intrusion labels are not equivalent to data-exfiltration labels.",
            "AgentDojo is a benchmark environment; event logs require instrumentation/export.",
            "Nearest-timestamp matching is weaker than exact session/trace identifiers.",
            "Dependence-discounted log-odds fusion is an approximation."
        ]
    },f,indent=2)

manifest={
    "framework":"JAMFUSE",
    "seed":SEED,
    "flow_rows_loaded":len(flow_df),
    "agent_events_loaded":len(agent_df),
    "aligned_rows":len(aligned_df),
    "aabm_metrics_available":bool(aabm_metrics_available),
    "ceif_metrics_available":bool(ceif_metrics_available),
    "controlled_simulation_enabled":bool(ENABLE_CONTROLLED_SIMULATION),
    "flow_features":len(flow_features)
}
with open(OUTPUT_DIR/"experiment_manifest.json","w") as f: json.dump(manifest,f,indent=2)
print("Completed. Outputs:")
for p in sorted(OUTPUT_DIR.iterdir()): print(p.name, round(p.stat().st_size/1024,1),"KB")

## Recommended paper reporting checklist
- Report the exact CIC-IDS2017 mirror/version, source files, class mapping, rows retained, and cleaning operations.
- Report AgentDojo version, suite, tasks, attack/defense configuration, model backend, trace collection procedure, and annotation policy.
- Keep scenario-level train/test splits separate to avoid the same agent scenario appearing in both partitions.
- For CEIF, report only explicitly labelled aligned exfiltration events and compare traffic-only, agent-only, and fused models on identical test sessions.
- Include confidence intervals, class imbalance, threshold-selection protocol, feature schema, hardware/runtime, and whether controlled simulations were used.
- Do not claim encrypted payload inspection: ETAF operates on flow metadata.